# Data Preparation

Parses the AMI Meeting Corpus XML files and produces three clean JSON datasets:
- `data/processed/importance_data.json` — sentence-level binary classification (important / not important)
- `data/processed/action_data.json` — sentence-level binary classification (action / not action)
- `data/processed/decision_data.json` — sentence-level binary classification (decision / not decision)

Also merges in any manually labeled examples from `data/labeled/`.

**Run this once before training any model.**

## 1. Setup

In [ ]:
import os
import json
import random
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

# ── Paths ──────────────────────────────────────────────────────────────────────
# Adjust ROOT if you're running from a different working directory
ROOT        = Path("../")          # project root (one level above notebooks/)
AMI_ROOT    = ROOT / "amicorpus"
LABELED_DIR = ROOT / "data" / "labeled"
OUT_DIR     = ROOT / "data" / "processed"

OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)

print(f"AMI corpus  : {AMI_ROOT}")
print(f"Labeled data: {LABELED_DIR}")
print(f"Output dir  : {OUT_DIR}")

: 

## 2. Parse AMI Abstractive XML Files

Each `.abssumm.xml` file contains four sections:
- `<abstract>` — general meeting summary sentences  
- `<actions>` — action items assigned to participants  
- `<decisions>` — decisions made during the meeting  
- `<problems>` — open problems / unresolved issues

In [ ]:
def parse_abssumm(xml_path: Path) -> dict:
    """
    Parse one .abssumm.xml file.
    Returns a dict with keys: abstract, actions, decisions, problems
    Each value is a list of sentence strings.
    """
    NS = "http://nite.sourceforge.net/"
    result = {"abstract": [], "actions": [], "decisions": [], "problems": []}

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except ET.ParseError:
        return result

    for tag in result:
        section = root.find(tag)
        if section is None:
            continue
        for sent in section.findall("sentence"):
            text = (sent.text or "").strip()
            if text:
                result[tag].append(text)

    return result


# Parse every .abssumm.xml file in the abstractive/ folder
abssumm_dir = AMI_ROOT / "abstractive"
xml_files   = sorted(abssumm_dir.glob("*.abssumm.xml"))

parsed = []
for f in xml_files:
    data = parse_abssumm(f)
    data["meeting"] = f.stem.replace(".abssumm", "")
    parsed.append(data)

print(f"Parsed {len(parsed)} meeting files")

# Quick sanity check on the first meeting
ex = parsed[0]
print(f"\nMeeting: {ex['meeting']}")
for k in ["abstract", "actions", "decisions", "problems"]:
    print(f"  {k}: {len(ex[k])} sentences")

## 3. Build Datasets

### Strategy

| Dataset | Positive examples | Negative examples |
|---|---|---|
| **importance** | `actions` + `decisions` sentences | `abstract` + `problems` sentences |
| **action** | `actions` sentences | `abstract` + `decisions` + `problems` sentences |
| **decision** | `decisions` sentences | `abstract` + `actions` + `problems` sentences |

Negatives are sampled to keep class balance (max 2:1 ratio).

In [ ]:
def build_classification_dataset(
    parsed_meetings: list,
    positive_keys: list,
    negative_keys: list,
    max_neg_ratio: float = 2.0,
    seed: int = 42,
) -> list:
    """
    Build a list of {text, label} dicts.
    label=1 for positive class, label=0 for negative class.
    """
    random.seed(seed)
    positives = []
    negatives = []

    for meeting in parsed_meetings:
        for key in positive_keys:
            for sent in meeting[key]:
                positives.append({"text": sent, "label": 1, "source": "ami"})
        for key in negative_keys:
            for sent in meeting[key]:
                negatives.append({"text": sent, "label": 0, "source": "ami"})

    # Cap negatives to avoid extreme imbalance
    max_neg = int(len(positives) * max_neg_ratio)
    if len(negatives) > max_neg:
        negatives = random.sample(negatives, max_neg)

    dataset = positives + negatives
    random.shuffle(dataset)
    return dataset


importance_data = build_classification_dataset(
    parsed,
    positive_keys=["actions", "decisions"],
    negative_keys=["abstract", "problems"],
)

action_data = build_classification_dataset(
    parsed,
    positive_keys=["actions"],
    negative_keys=["abstract", "decisions", "problems"],
)

decision_data = build_classification_dataset(
    parsed,
    positive_keys=["decisions"],
    negative_keys=["abstract", "actions", "problems"],
)

for name, ds in [("importance", importance_data), ("action", action_data), ("decision", decision_data)]:
    c = Counter(d["label"] for d in ds)
    print(f"{name:12s}: {len(ds):4d} examples  (pos={c[1]}, neg={c[0]})")

## 4. Merge Manual Labels

Any examples in `data/labeled/` are added to the corresponding dataset.
These are small high-quality examples labeled by hand.

In [ ]:
def load_labeled(path: Path) -> list:
    """Load a labeled JSON file, stripping internal _note fields."""
    if not path.exists():
        return []
    with open(path) as f:
        items = json.load(f)
    # Keep only text + label, add source tag
    clean = []
    for item in items:
        if "text" in item and "label" in item:
            clean.append({"text": item["text"], "label": item["label"], "source": "manual"})
    return clean


manual_importance = load_labeled(LABELED_DIR / "importance_labels.json")
manual_decision   = load_labeled(LABELED_DIR / "decision_labels.json")
# action_labels.json uses BIO tokens, not suitable for binary classification here

importance_data = manual_importance + importance_data
decision_data   = manual_decision   + decision_data

print(f"After merging manual labels:")
for name, ds in [("importance", importance_data), ("action", action_data), ("decision", decision_data)]:
    c = Counter(d["label"] for d in ds)
    print(f"  {name:12s}: {len(ds):4d} examples  (pos={c[1]}, neg={c[0]})")

## 5. Save to Disk

In [ ]:
def save_json(data: list, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved {len(data):4d} examples → {path}")


save_json(importance_data, OUT_DIR / "importance_data.json")
save_json(action_data,     OUT_DIR / "action_data.json")
save_json(decision_data,   OUT_DIR / "decision_data.json")

## 6. Quick Inspection

In [ ]:
print("=== Sample importance examples ===")
for ex in random.sample(importance_data, min(4, len(importance_data))):
    label_str = "IMPORTANT" if ex["label"] == 1 else "not important"
    print(f"  [{label_str:13s}] {ex['text'][:90]}")

print("\n=== Sample action examples ===")
for ex in random.sample(action_data, min(4, len(action_data))):
    label_str = "ACTION" if ex["label"] == 1 else "not action"
    print(f"  [{label_str:10s}] {ex['text'][:90]}")

print("\n=== Sample decision examples ===")
for ex in random.sample(decision_data, min(4, len(decision_data))):
    label_str = "DECISION" if ex["label"] == 1 else "not decision"
    print(f"  [{label_str:12s}] {ex['text'][:90]}")